In [1]:
import sys
sys.path.append('/data/taylor_group/Nima/ATLAS')
sys.path.append('../')

import json
import pickle
import copy 
import random 
import glob

from tqdm.auto import trange
import tqdm_joblib
from joblib import Parallel, delayed

import numpy as np
import jax.numpy as jnp
import jax.random as jrandom
import jax
jax.config.update("jax_enable_x64", True)

import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from numpyro.distributions import constraints

from jug.engine.session import TimingSession

from ATLAS.data import PTA_Data
from ATLAS.signals.factorized.base import Red, SuperSignal
from ATLAS.signals.correlated.base import Correlated
from ATLAS import parameterized
from ATLAS.signals.timing.base import build_multi_psr_timing_model, MultiPsrTimingModel
from ATLAS.signals.timing.base import setup_timing_model
from ATLAS.nMatrix.base import WhiteCov
from ATLAS.signals import signals_utils as sutils
import ATLAS.utils as atlas_utils
import ATLAS.signals.factorized.utils as unc_utils
import ATLAS.signals.correlated.utils as cor_utils
from ATLAS import pulsar

# import MultiHMCGibbs

from ATLAS.samplers.canetoadracing import MultiHMCGibbs, MultiHMCGibbsWithAnalytic, AnalyticRhoTransition

%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
hist_settings = dict(
    bins = 'auto',
    histtype = 'step', 
    lw = 3,
    density = True
)

plt.style.use('default')
def figsize(scale, wc = 1, hc = 1):
    fig_width_pt = 513.17 #469.755                  # Get this from LaTeX using \the\textwidth
    inches_per_pt = 1.0/72.27                       # Convert pt to inch
    golden_mean = (np.sqrt(5.0)-1.0)/2.0            # Aesthetic ratio (you could change this)
    fig_width = fig_width_pt*inches_per_pt*scale    # width in inches
    fig_height = fig_width*golden_mean              # height in inches
    fig_size = [wc * fig_width,hc * fig_height]
    return fig_size
plt.rcParams.update(plt.rcParamsDefault)

params = {#'backend': 'pdf',
        'axes.labelsize': 12,
        'lines.markersize': 4,
        'font.size': 10,
        'xtick.major.size':6,
        "xtick.top": True,
        "ytick.right": True,
        "xtick.minor.visible": True,
        "xtick.major.top": True, 
        "xtick.minor.top": True,
        "ytick.minor.visible": True, 
        "ytick.major.right": True, 
        "ytick.minor.right": True,
        "ytick.direction": "in",
        "xtick.direction": "in",
        'xtick.minor.size':3,  
        'ytick.major.size':6,
        'ytick.minor.size':3, 
        'xtick.major.width':0.5,
        'ytick.major.width':0.5,
        'xtick.minor.width':0.5,
        'ytick.minor.width':0.5,
        'lines.markeredgewidth':1,
        'axes.linewidth':1.2,
        'legend.fontsize': 7,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'savefig.dpi':200,
        'path.simplify':True,
        'font.family': 'serif',
        'font.serif':'Times',
        # "text.usetex": True,
        #'text.latex.preamble': [r'\usepackage{amsmath}'],
        'text.usetex':False,
        'figure.figsize': figsize(0.5, 1, 1)}
plt.rcParams.update(params)
plt.rcParams['font.family'] = 'STIXGeneral'  # Closely matches Computer Modern
plt.rcParams['mathtext.fontset'] = 'stix'    # Use STIX for math

# Loading data

## Option 1: To test the full capability of `ATLAS`, we only need par and tim files.

### We are using the `enterprise` pulsar object. This will be replaced by an in-house object very soon.

In [3]:
from enterprise.pulsar import Pulsar

In [4]:
pnames = ['B1855+09', 'B1937+21', 'B1953+29', 'J0023+0923', 'J0030+0451', 'J0340+4130', 'J0406+3039', 
'J0437-4715', 'J0509+0856', 'J0557+1551', 'J0605+3757', 'J0610-2100', 'J0613-0200', 'J0636+5128', 
'J0645+5158', 'J0709+0458', 'J0740+6620', 'J0931-1902', 'J1012+5307', 'J1012-4235', 'J1022+1001', 
'J1024-0719', 'J1125+7819', 'J1312+0051', 'J1453+1902', 'J1455-3330', 'J1600-3053', 'J1614-2230', 
'J1630+3734', 'J1640+2224', 'J1643-1224', 'J1705-1903', 'J1713+0747', 
'J1719-1438', 'J1730-2304', 
'J1738+0333', 'J1741+1351', 'J1744-1134', 'J1745+1017', 'J1747-4036', 'J1751-2857', 'J1802-2124', 
'J1811-2405', 'J1832-0836', 'J1843-1113', 'J1853+1303', 'J1903+0327', 'J1909-3744', 'J1910+1256', 
'J1911+1347', 'J1918-0642', 'J1923+2515', 'J1944+0907', 'J1946+3417', 'J2010-1323', 'J2017+0603', 
'J2033+1734', 'J2043+1711', 'J2124-3358', 'J2145-0750', 'J2214+3000', 'J2229+2643', 'J2234+0611', 
'J2234+0944', 'J2302+4442', 'J2317+1439', 'J2322+2057']
len(pnames)

67

In [5]:
parfiles_ref = sorted(glob.glob('../data/NG15/partim/par/*.par'))
timfiles_ref = sorted(glob.glob('../data/NG15/partim/tim/*.tim'))

### Filter the par and tim files to remove "ao-only" files

In [6]:
parfiles = []
for pname in pnames:
    for p in parfiles_ref:
        if pname in p and not 'ao' in p and not 'gbt' in p:
            parfiles.append(p)

timfiles = []
for pname in pnames:
    for p in timfiles_ref:
        if pname in p and not 'ao' in p and not 'gbt' in p:
            timfiles.append(p)

In [7]:
assert len(parfiles) == len(pnames)

In [9]:
Npulsars = len(pnames) # how many pulsars do you want?
Npulsars

67

In [10]:
def doit(pidx):
    return Pulsar(parfiles[pidx], timfiles[pidx], sort=True, ephem='DE440', timing_package='PINT')
with tqdm_joblib.tqdm_joblib(desc="PINT Loading...", total=len(parfiles)) as progress_bar:
    psrs = Parallel(n_jobs=24)(delayed(doit)(i) for i in range(len(parfiles)))
    # psrs = Parallel(n_jobs=24)(delayed(doit)(i) for i in range(1))

with open('../data/NG15/NG15_v1p1_final_pint_psrs_sorted.pkl', 'wb') as fout:
    pickle.dump(psrs, fout)

PINT Loading...:   0%|          | 0/67 [00:00<?, ?it/s]

2026-06-30 14:38:51.897 | DEBUG    | pint.toa:get_TOAs:211 - Using CLOCK = BIPM2019 from the given model
2026-06-30 14:38:51.925 | DEBUG    | pint.toa:get_TOAs:211 - Using CLOCK = BIPM2019 from the given model
2026-06-30 14:38:51.927 | DEBUG    | pint.toa:get_TOAs:211 - Using CLOCK = BIPM2019 from the given model
2026-06-30 14:38:51.937 | DEBUG    | pint.toa:get_TOAs:211 - Using CLOCK = BIPM2019 from the given model
2026-06-30 14:38:51.941 | DEBUG    | pint.toa:get_TOAs:211 - Using CLOCK = BIPM2019 from the given model
2026-06-30 14:38:51.964 | DEBUG    | pint.toa:get_TOAs:211 - Using CLOCK = BIPM2019 from the given model
2026-06-30 14:38:51.964 | DEBUG    | pint.toa:get_TOAs:211 - Using CLOCK = BIPM2019 from the given model
2026-06-30 14:38:51.971 | DEBUG    | pint.toa:get_TOAs:211 - Using CLOCK = BIPM2019 from the given model
2026-06-30 14:38:51.972 | DEBUG    | pint.toa:get_TOAs:211 - Using CLOCK = BIPM2019 from the given model
2026-06-30 14:38:51.972 | DEBUG    | pint.toa:get_TOAs:

In [11]:
with open('../data/NG15/NG15_v1p1_final_pint_psrs_sorted.pkl', 'rb') as fin:
    psrs = pickle.load(fin)
psrs = [psr for psr in psrs if psr.name != 'J1713+0747'][:2]

In [12]:
with open('../data/NG15/15yr_wn_dict.json', 'r') as fin:
    noise_dict = json.load(fin)

In [13]:
chosen_pidx = 0 #np.argmin([len(psr.toas) for psr in psrs])
chosen_pidx 

0

In [14]:
psrs = psrs[chosen_pidx :chosen_pidx +1]

In [15]:
len(psrs)

1

In [16]:
psrs[0].name

'B1855+09'

In [17]:
timfiles[chosen_pidx :chosen_pidx +1]

['../data/NG15/partim/tim/B1855+09_PINT_20220301.nb.tim']

## Option 2: If you do not want to vary the timing model params, use the pre-made 15 year pulsar objects.

In [4]:
from enterprise.pulsar import FeatherPulsar
import glob

# feather_files = glob.glob('/data/taylor_group/Nima/15yr_stochastic_analysis/tutorials/data/feathers/*.feather')
feather_files = glob.glob('../data/NG15/feathers/*.feather')
psrs = []
for feather_file in feather_files:
    psr = FeatherPulsar.read_feather(feather_file)
    psrs.append(psr)

# with open('/data/taylor_group/Nima/15yr_stochastic_analysis/tutorials/data/15yr_wn_dict.json', 'r') as fin:
#     noise_dict = json.load(fin)
with open('../data/NG15/15yr_wn_dict.json', 'r') as fin:
    noise_dict = json.load(fin)

FeatherPulsar.read_feather: cannot find fitpars in feather file ../data/NG15/feathers/v1p1_de440_pint_bipm2019-J1312+0051.feather.
FeatherPulsar.read_feather: cannot find setpars in feather file ../data/NG15/feathers/v1p1_de440_pint_bipm2019-J1312+0051.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file ../data/NG15/feathers/v1p1_de440_pint_bipm2019-J1312+0051.feather.
FeatherPulsar.read_feather: cannot find fitpars in feather file ../data/NG15/feathers/v1p1_de440_pint_bipm2019-J0613-0200.feather.
FeatherPulsar.read_feather: cannot find setpars in feather file ../data/NG15/feathers/v1p1_de440_pint_bipm2019-J0613-0200.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file ../data/NG15/feathers/v1p1_de440_pint_bipm2019-J0613-0200.feather.
FeatherPulsar.read_feather: cannot find fitpars in feather file ../data/NG15/feathers/v1p1_de440_pint_bipm2019-J0610-2100.feather.
FeatherPulsar.read_feather: cannot find setpars in feather file ../data/NG15/feathers

In [5]:
psrs = psrs[0:2]

In [6]:
print(psrs)

[<Pulsar J1312+0051: 1705 res, 62 pars>, <Pulsar J0613-0200: 17124 res, 193 pars>]


In [7]:
data = PTA_Data(psrs, 
                fixed_res = False,
                marg_timing=False, 
                diag_white_cov=False,
                linear_timing = True)

# Model Construction

## White Noise Model

In [8]:
wn_model = WhiteCov(data = data)

  0%|          | 0/2 [00:00<?, ?it/s]

## Timing Model

In [9]:
# SAMPLE = ['F0', 'F1'] #psrs[0].fitpars[1:] #['F0', 'F1']
# tm_model = build_multi_psr_timing_model(parfiles[chosen_pidx :chosen_pidx + 1], 
#                                         timfiles[chosen_pidx :chosen_pidx + 1], 
#                                         SAMPLE, 
#                                         load_how_many_in_parallel = 24,
#                                         data = data)
# # data.add_timing_design_matrix(tm_model.Mmats)
# svd_norm_mmat = _timing_model_svd(psrs[0].Mmat)
# data.add_timing_design_matrix([svd_norm_mmat])

In [10]:
raw_res = jnp.concat(data.raw_residuals)

### Initial white noise guess

In [11]:
theta0_wn = wn_model.prior_draw()
theta0_wn = wn_model.params_dict_to_vector(noise_dict)
theta0_wn[:5]

Array([ 1.04982172,  0.96948031, -6.17884779, -5.56093065, -7.03963453],      dtype=float64)

## Red Noise Model

In [12]:
non_gwb_nfreqs = 30

### Non-GWB Model

#### Choose your PSD model

In [13]:
# Example:
# chosen_psd_model_nongwb, helper_dictionary_nongwb = unc_utils.varied_gamma_pl(renorm_const = 1)
chosen_psd_model_nongwb, helper_dictionary_nongwb = unc_utils.spectrum(renorm_const = 1, crn_bins = non_gwb_nfreqs)

In [14]:
sig_unc = Red(name='unc', 
                nfreqs=non_gwb_nfreqs, 
                halflog10_rho_range=(-9,-2),
                data = data,
                use_pulsar_tspan = False)

### GWB Model

#### Choose your PSD model

In [15]:
gwb_nfreqs = 14

In [16]:
chosen_psd_model, chosen_orf_model, gwb_helper_dictionary = cor_utils.fixed_gamma_hd_pl(renorm_const = 1)

In [17]:
sig_cor = Correlated(name='cor', 
                    nfreqs=gwb_nfreqs, 
                    halflog10_rho_range=(-9,-2),
                    data = data)

### The user should not see this in the final version, but for now you need to see this!

In [18]:
signal_helper = {
    'shared_basis': { #shared basis does not mean the nfreqs are the same between signals.
                    #It means that one basis includes the other one.
    'signal_list': [sig_unc, sig_cor], 
    'index_of_signal_used_for_basis': 0,
    },

    'separate': {
        'signal_list': []
        },
    'order': 'unc,cor',
    }

In [19]:
sig = SuperSignal(signal_helper = signal_helper, 
                data=data,
)

### `sig.get_helpers` updates the white noise covaraince matrix

In [20]:
TNT, TNr, rNr, logdetN = sig.get_helpers(reff = raw_res, white_noise_params=theta0_wn)
helpers = (TNT, TNr, rNr, logdetN)
TNT.shape, TNr.shape, rNr.shape, logdetN.shape

((2, 253, 253), (2, 253), (), ())

In [21]:
ll, ul = wn_model.get_prior_bounds() #white noise prior bounds
ll[:5], ul[:5]

(Array([ 0.01,  0.01, -9.  , -9.  , -9.  ], dtype=float64),
 Array([10., 10., -5., -5., -5.], dtype=float64))

In [22]:
sig_unc.tspans/(86400 * 365.25)

15.351390925823674

In [23]:
gwb_Tspan = data.pta_tspan
gwb_Tspan/(365.25 * 86400)

15.351390925823674

In [24]:
o = parameterized.MultiPulsarRedNoise(
            # ---- GWB ----
            gwb_psd_func = chosen_psd_model,
            orf_func = chosen_orf_model,
            crn_bins = gwb_nfreqs,
            int_bins = non_gwb_nfreqs,
            f_common = sig_cor.freqs, 
            f_intrin = sig_unc.freqs,
            df = 1/gwb_Tspan,
            Tspan = gwb_Tspan, 
            Npulsars = sig.npsrs,
            psr_pos = data.psr_pos,
            gwb_helper_dictionary = gwb_helper_dictionary,
            renorm_const = 1,
            
            # ---- non-GWB intrinsic red noise (optional) ----
            irn_psd_func=chosen_psd_model_nongwb,
            irn_helper_dictionary=helper_dictionary_nongwb,

            # dm_psd_func=chosen_psd_model_nongwb,
            # dm_helper_dictionary=helper_dictionary_nongwb,
            # f_dm = sig_unc.freqs,
            # dm_bins = len(sig_unc.freqs)
            )
sig.add_parameterization(o)

In [25]:
x0 = o.make_initial_guess(jrandom.key(1273442)) #red noise model params
phi0 = o.get_phi_mat_full(x0) #red noise cov matrix
phi0.shape, x0.shape

((30, 2, 2), (61,))

# Likelihood

### Reparameterized likelihood

In [26]:
z0 = jrandom.normal(jrandom.key(10098), shape = (sig.npsrs, sig.nmodes)) # re-parameterized coefficients

In [27]:
z0.shape

(2, 253)

In [28]:
lnprob0, coeff0 = sig.lnposterior_reparam(helpers = helpers, 
                                            red_params = x0, 
                                            z = z0)
lnprob0

Array(242610.3767119, dtype=float64)

In [29]:
%timeit sig.lnposterior_reparam(helpers = helpers,  red_params = x0, z = z0)

1.67 ms ± 65.6 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [30]:
red_params = x0
z = z0
the_object = sig

In [31]:
# TNT, TNr, rNr, logdet_N = helpers
# red_noise_cov = the_object.model.get_phi_mat_full(red_params)
# if the_object.npsrs == 1:
#     phiinvs_diags = jnp.repeat(1/red_noise_cov, 2, axis = 0) #[nmodes, npsrs]
#     logdet_phimat = 2 * jnp.sum(jnp.log(red_noise_cov)) #2 is to account for 2*nfreq=nmodes
# else:
#     phiinvs, logdet_phimat = the_object.model.get_phi_mat_inv(red_noise_cov)
#     phiinvs_diags = phiinvs.diagonal(axis1 = -2, axis2 = -1) #[nmodes, npsrs]

# if the_object.linear_timing and not the_object.marg_tm:
#     phiinvs_diags_ltm = jnp.full(shape = (the_object.nmodes, the_object.npsrs), fill_value = the_object.lowest_value_eq_to_zero)
#     phiinvs_diags = phiinvs_diags_ltm.at[the_object.linear_timing_model_size:, :].add(phiinvs_diags)

# # Posterior precision Cholesky (cho_factor equivalent), batched over pulsars
# Sigma_inv = TNT.at[:, the_object._diag_idx , the_object._diag_idx ].add(phiinvs_diags.T)     # [npsr, nmodes, nmodes]
# Sigma_inv_L = jsl.cho_factor(Sigma_inv, lower = True)  # [npsr, nmodes, nmodes]

# # MAP coefficients via cho_solve pattern: forward then back substitution
# a_hat = jsl.cho_solve(Sigma_inv_L, TNr[..., None])

# # Standardizing transform via back substitution
# Lz = jax.lax.linalg.triangular_solve(
#     Sigma_inv_L[0], z[..., None], left_side=True, lower=True, transpose_a=True,
# )  # L^T Lz = z

# coeff = a_hat + Lz  # [npsr, nmodes, 1]

# lndet_Jac = -jnp.sum(jnp.log(Sigma_inv_L[0].diagonal(axis1=-2, axis2=-1)))

# # Log-likelihood
# aFNr      = jnp.sum(coeff[..., 0] * TNr)
# aFNFa     = jnp.sum(coeff.mT @ TNT @ coeff)
# lnlike_value = aFNr - 0.5 * aFNFa

# if the_object.npsrs == 1:
#     lnprior_value = -0.5 * ((coeff[:, the_object.linear_timing_model_size:, 0]**2 * phiinvs_diags[the_object.linear_timing_model_size:, :].T).sum() + logdet_phimat)
# else:
#     aG = coeff[:, the_object.linear_timing_model_size:] #[npsr, 2 * nfreq, 1]
#     lnprior_value = -0.5 * ((aG.transpose(1, 2, 0) @ phiinvs @ aG.transpose(1, 0, 2)).sum() + logdet_phimat)

# ans =lnlike_value + lnprior_value + lndet_Jac - 0.5 * (rNr + logdet_N)

In [32]:
padded_ndxs = jnp.where(TNr[0] == 0.)[0]
# print(padded_ndxs)

In [33]:
def model():

    z = numpyro.sample('z_red', dist.Normal().expand([sig.npsrs, sig.nmodes]))

    rn_params = numpyro.sample('rn_params', dist.Uniform(o.lower_prior_lim_all, o.upper_prior_lim_all))

    lnpost_val, a = sig.lnposterior_reparam(helpers=helpers, red_params=rn_params, z=z)

    # numpyro.deterministic('a', a)
    numpyro.factor('lnpost', lnpost_val + 0.5 * jnp.sum(z**2))
    # numpyro.factor('first_correction', -0.5 * jnp.sum(z[0, padded_ndxs]))

In [34]:
from numpyro.infer.initialization import init_to_value
init_dict = {'z_red': jnp.zeros((sig.npsrs, sig.nmodes))}

In [35]:
nuts_kernel = numpyro.infer.NUTS(model=model,
                                 # init_strategy=init_to_value(values=init_dict),
                                 )
mcmc = numpyro.infer.MCMC(sampler=nuts_kernel,
                          num_warmup=1000,
                          num_samples=4000,
                          num_chains=1,
                          )
mcmc.run(jrandom.key(150914))
atlas_helpers_samples = mcmc.get_samples()

sample: 100%|██████████| 5000/5000 [18:58<00:00,  4.39it/s, 31 steps of size 1.04e-01. acc. prob=0.92]   


In [32]:
# plt.hist(atlas_helpers_samples['rn_params'][:, -1], **hist_settings)
# plt.show()

In [ ]:
# %timeit lnprob0, coeff0 = sig.lnposterior_reparam(helpers = (TNT, TNr), red_params = x0, z = z0)

In [54]:
from enterprise_extensions import blocks
from enterprise.signals import signal_base, gp_signals
from enterprise.pulsar import Pulsar
from astropy.time import TimeDelta
from pint.simulation import make_fake_toas_fromMJDs
import astropy.units as u
from pint.models import get_model

from enterprise_extensions import sampler
from enterprise.signals import parameter
from enterprise.signals import gp_priors

In [55]:
Tspan = gwb_Tspan

In [56]:
# s = gp_signals.MarginalizingTimingModel(use_svd=True, normed = True)
s = gp_signals.TimingModel(use_svd=True)
s += blocks.white_noise_block(
    vary=True,
    inc_ecorr=True,
    gp_ecorr=False,
    select='backend',
    tnequad=False,
)

# log10_A = parameter.Uniform(-18., -11.)('log10A')
# gamma = parameter.Uniform(0., 7.)('gamma')

# s+= gp_signals.FourierBasisGP(
#     spectrum=gp_priors.powerlaw(log10_A=log10_A, gamma = gamma),
#     modes = sig_unc.freqs,
#     # components = len(sig_unc.freqs),
#     Tspan=Tspan,
#     name="red_noise",
# )
log10_rho = parameter.Uniform(-9., -2., size=len(sig_unc.freqs))
s+= gp_signals.FourierBasisGP(
    spectrum=gp_priors.free_spectrum(log10_rho=log10_rho),
    components=len(sig_unc.freqs),
    modes = sig_unc.freqs,
    Tspan=Tspan,
    # name="red_noise",
)

pta = signal_base.PTA(
    [s(p) for p in psrs], signal_base.LogLikelihoodDenseCholesky
)


In [57]:
p0 = pta.map_params(np.hstack([p.sample() for p in pta.params]))

In [60]:
pta.get_TNr(noise_dict)[0][:10]

array([13.59257304,  5.56881285,  1.48093944, -1.72209773,  3.16839839,
       -1.30959348, -0.86459357, -0.83741256, -8.28529988, -1.76854608])

In [70]:
single_helpers_dict = {'single_TNT': TNT[0], 'single_TNr': TNr[0]}

with open('single_psr_padded_helpers.pkl', 'wb') as f:
    pickle.dump(single_helpers_dict, f)

In [66]:
TNr_not_padded = TNr[0][jnp.where(TNr[0] != 0.)[0]]

In [68]:
TNr_not_padded / pta.get_TNr(noise_dict)[0]

Array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.], dtype=float64)

In [50]:
pta.get_TNT(noise_dict)[0][:10]

array([[ 3.78467372e+10,  1.15156009e+10, -5.50823273e+08, ...,
         9.48088630e+10,  2.16289663e+11, -2.08434508e+11],
       [ 1.15156009e+10,  3.25337451e+10,  5.04756775e+09, ...,
         1.98563359e+11,  5.82792122e+10, -2.22033467e+11],
       [-5.50823273e+08,  5.04756775e+09,  3.76518688e+10, ...,
         1.28898452e+11, -2.84999255e+10, -1.73423030e+11],
       ...,
       [ 1.87921353e+09,  7.59729076e+08, -4.95107068e+05, ...,
        -2.56191505e+11, -1.99132680e+11, -6.23937180e+10],
       [-1.04178920e+10, -8.16284522e+09, -5.94873684e+09, ...,
        -2.78273967e+11, -1.44817606e+11,  2.15094302e+11],
       [ 6.28916114e+09,  2.71825040e+09, -3.68361760e+09, ...,
        -2.71331710e+11, -1.66933729e+11,  2.71472685e+10]],
      shape=(10, 122))

In [51]:
TNT[0][:10]

Array([[ 3.78467372e+10,  1.15156009e+10, -5.50823273e+08, ...,
         9.48088630e+10,  2.16289663e+11, -2.08434508e+11],
       [ 1.15156009e+10,  3.25337451e+10,  5.04756775e+09, ...,
         1.98563359e+11,  5.82792122e+10, -2.22033467e+11],
       [-5.50823273e+08,  5.04756775e+09,  3.76518688e+10, ...,
         1.28898452e+11, -2.84999255e+10, -1.73423030e+11],
       ...,
       [ 1.87921353e+09,  7.59729076e+08, -4.95107068e+05, ...,
        -2.56191505e+11, -1.99132680e+11, -6.23937180e+10],
       [-1.04178920e+10, -8.16284522e+09, -5.94873684e+09, ...,
        -2.78273967e+11, -1.44817606e+11,  2.15094302e+11],
       [ 6.28916114e+09,  2.71825040e+09, -3.68361760e+09, ...,
        -2.71331710e+11, -1.66933729e+11,  2.71472685e+10]],      dtype=float64)

In [43]:
pta.get_basis()[0][0, :]

array([-5.06580790e-02, -2.66636364e-02,  4.45429749e-02, -4.20352625e-02,
        1.97848715e-02, -1.67117822e-02,  1.40674368e-02, -2.05133754e-02,
       -4.27068260e-02,  6.37651157e-03,  4.68966120e-02,  3.06671092e-02,
        1.20595234e-03, -6.38453342e-03,  1.34314714e-02,  7.78381422e-04,
        8.80838877e-03,  9.52062172e-03,  1.72482397e-02,  2.23223684e-03,
       -2.11336108e-02,  7.01661717e-03,  1.26462634e-02,  4.10764819e-03,
        1.70179303e-02,  2.35354148e-02, -1.45566267e-02, -1.77049759e-02,
        1.26705812e-04, -3.33073309e-02, -2.50582783e-02,  2.41219688e-02,
        1.33769829e-02, -1.19245638e-02,  8.97289103e-03, -3.79112659e-03,
        1.33252081e-02,  9.12060571e-03,  1.00824023e-02,  2.15627201e-02,
        5.94580598e-03,  2.58496548e-02, -1.16561480e-02,  1.10853367e-02,
       -5.90532862e-02, -2.95395986e-02, -8.68790776e-03,  1.41016436e-04,
       -8.54795909e-03,  8.67217742e-03,  4.62576242e-02,  6.24074818e-02,
        5.24993644e-02, -

In [47]:
sig.get_Fmat_concat[0]

Array([-5.06580790e-02, -2.66636364e-02,  4.45429749e-02, -4.20352625e-02,
        1.97848715e-02, -1.67117822e-02,  1.40674368e-02, -2.05133754e-02,
       -4.27068260e-02,  6.37651157e-03,  4.68966120e-02,  3.06671092e-02,
        1.20595234e-03, -6.38453342e-03,  1.34314714e-02,  7.78381422e-04,
        8.80838877e-03,  9.52062172e-03,  1.72482397e-02,  2.23223684e-03,
       -2.11336108e-02,  7.01661717e-03,  1.26462634e-02,  4.10764819e-03,
        1.70179303e-02,  2.35354148e-02, -1.45566267e-02, -1.77049759e-02,
        1.26705812e-04, -3.33073309e-02, -2.50582783e-02,  2.41219688e-02,
        1.33769829e-02, -1.19245638e-02,  8.97289103e-03, -3.79112659e-03,
        1.33252081e-02,  9.12060571e-03,  1.00824023e-02,  2.15627201e-02,
        5.94580598e-03,  2.58496548e-02, -1.16561480e-02,  1.10853367e-02,
       -5.90532862e-02, -2.95395986e-02, -8.68790776e-03,  1.41016436e-04,
       -8.54795909e-03,  8.67217742e-03,  4.62576242e-02,  6.24074818e-02,
        5.24993644e-02, -

# Timing Model

### `z_tm` is the value by which the deviations from the best fit are changed. This is an inferable parameter

In [ ]:
z_tm = {}
for pidx in range(sig.npsrs):
    z_tm.update({f'z_{k};{pidx}': 100 * jrandom.normal(jrandom.key(random.randint(0, 16162))) for k in SAMPLE})
z_tm 

In [ ]:
z_tm_concat = jnp.array(list(z_tm.values()))
assert z_tm_concat.shape[0] == len(psrs) * len(SAMPLE)

In [ ]:
tm_model.residuals(z_tm_concat),tm_model.residuals(z_tm_concat).shape

In [ ]:
# %timeit tm_model.residuals(z_tm_concat)

In [ ]:
# z_tm_concat = tm_model.get_epsilon(helpers = (MNM, MNr), key = jrandom.key(12))

In [ ]:
# tm_model.linear_residuals(z_tm_concat)

In [ ]:
tm_param_dim = len(z_tm_concat)
tm_param_dim 

# Sampling

### NOTE: Comment out the parts of the model you do not want to sample

### NOTE: If you want to fix the white noise to the NG15 noise dictionary, make sure you use the white noise dictionary to make the TNT, TNr matricies instead of prior draws.

In [ ]:
def model():

    ######################################## Timing Model ########################################
    # Declare the variables in the model
    z_tm_concat = numpyro.sample('z_tm_concat', dist.Normal(0, 100).expand([tm_param_dim]))
    stochastic_res = tm_model.residuals(z_tm_concat)
    ######################################## White Noise ########################################
    # Declare the variables in the model
    theta_wn = numpyro.sample('theta_wn', dist.Uniform(ll, ul))

    # Update the white noise covaraince matrix
    TNT, TNr, rNr, logdet_N = sig.get_helpers(reff = stochastic_res, 
                                              white_noise_params=theta_wn
                                             )

    # White noise direct contribution to the PTA likelihood
    numpyro.factor('lnpost_white', -0.5 * (rNr[0] + logdet_N))

    ######################################## Red Noise ########################################
    # Declare the variables in the model
    xs = numpyro.sample('xs', dist.Uniform(o.lower_prior_lim_all, o.upper_prior_lim_all)) #PSD params
    z_a = numpyro.sample('z_a', dist.Normal(0, 1).expand((sig.npsrs, sig.nmodes))) #the reparam coefficients

    # evaluate the posterior
    lprob, coeff = sig.lnposterior_reparam(helpers = (TNT, TNr), red_params = xs, z = z_a)
    numpyro.factor('lnpost', lprob)

    numpyro.deterministic('coeff', coeff)

## Not Block Gibbs

In [ ]:
mcmc_bad = MCMC(
    NUTS(model),
    num_warmup=1000,
    num_samples=4000,
    num_chains=1,
    # chain_method=chain_method,
    progress_bar=True
)
mcmc_bad.run(jrandom.key(100))

In [ ]:
samples = mcmc_bad.get_samples()['xs']
samples.shape

In [ ]:
for fidx in range(30):
    plt.hist(samples[:, fidx], **hist_settings)
    plt.show()

## Block Gibbs

In [ ]:
kernel = MultiHMCGibbsWithAnalytic(
        [
        NUTS(model), 
        AnalyticRhoTransition(model = model,
                            posterior_draw_func = tm_model.get_epsilon,
                            prior_draw_func = epsilon_prior_draw
                            rho_site  = "epsilons",
                            coeff_site = "coeff",
                            rho_low  = -9.0,
                            rho_high = -2.0)
        ],
        
        [
        ['z_tm_concat', 'z_a'],
        ['xs']
        ]
    )

kernel = NUTS(model, max_tree_depth=8, target_accept_prob=0.8, dense_mass=False)

In [ ]:
mcmc_bad = MCMC(
    kernel,
    num_warmup=1000,
    num_samples=4000,
    num_chains=1,
    # chain_method=chain_method,
    progress_bar=True
)
mcmc_bad.run(jrandom.key(100))

In [ ]:
samples = mcmc_bad.get_samples()['xs']
samples.shape

In [ ]:
samples[:, -1]

In [ ]:
plt.plot(samples[:, -1])
plt.show()

# ENTERPRISE

In [ ]:
from enterprise_extensions.gibbs_sampling import gibbs

In [ ]:
m = gibbs.BayesPowerSingle(psr=psrs[0],
        Tspan=None,
        select="backend",
        white_vary=True,
        inc_ecorr=True,
        ecorr_type="kernel",
        noise_dict=noise_dict,
        tm_marg=False,
        freq_bins=30,
        tnequad=False,
        log10rhomin=-9.0,
        log10rhomax=-4.0)

In [ ]:
# good_idxs = [psrs[0].fitpars.index('F0'), psrs[0].fitpars.index('F1')] + m.gwid[0].tolist()
good_idxs = [psrs[0].fitpars.index(x) for x in SAMPLE] + m.gwid[0].tolist()
good_idxs = np.array(good_idxs)
# good_idxs = np.arange(m.Tmat.shape[-1])
good_idxs

In [ ]:
keep_idxs = np.ix_(good_idxs, good_idxs)

In [ ]:
# m = gibbs.BayesPowerSingle(psr=psrs[0],
#         Tspan=None,
#         select="backend",
#         white_vary=False,
#         inc_ecorr=True,
#         ecorr_type="kernel",
#         noise_dict=noise_dict,
#         tm_marg=False,
#         freq_bins=30,
#         tnequad=False,
#         log10rhomin=-9.0,
#         log10rhomax=-4.0,
#         keep_coeff_idxs = keep_idxs)

In [ ]:
m.sample(
        niter=int(1e4),
        wniters=30,
        eciters=10,
        savepath='/data/taylor_group/Nima/PandorasBox/Trash/GibbsTrashCan',
        SCAMweight=30,
        AMweight=15,
        DEweight=50,
        covUpdate=1000,
        burn=10000,)

In [ ]:
chain = np.load('/data/taylor_group/Nima/PandorasBox/Trash/GibbsTrashCan/chain_1.npy', mmap_mode = 'r')
chain.shape

In [ ]:
chain = chain[int(0.25 * chain.shape[0]):, :]
chain.shape

In [ ]:
for fidx in range(5):
    plt.plot(chain[:, fidx])
    plt.show()

In [ ]:
chain.shape

In [ ]:
60 + 30 +  tm_model.Mmats[0].shape[-1]

In [ ]:
for fidx in range(30):
    plt.hist(chain[:, fidx], **hist_settings, label = 'ENTERPRISE')
    # plt.hist(samples[:, fidx], **hist_settings, label = 'ATLAS')
    # plt.title(m.pta.param_names[fidx])
    plt.legend()
    plt.show()

# Flows

In [ ]:
from Atlas.flows import Flow

In [ ]:
mm = Flow(data = jnp.array(chain),
        flow_num_layers=4,
        hidden_size=128,
        mlp_num_layers=2,
        num_bins=8,
        learning_rate=1e-4,
        B=6.0,
        p=0.3,
        seed=0
         )

In [ ]:
mm.fit()

In [ ]:
gen_samps = mm.sample((int(1e4)))

In [ ]:
plt.hist(gen_samps[:, 23], **hist_settings)
plt.hist(chain[:, 23], **hist_settings)
plt.show()

In [ ]:
z = mm.backward_pass(gen_samps)
z.shape

In [ ]:
plt.hist(z[:, 23], **hist_settings)
plt.show()

In [ ]:
chain.shape[-1]

In [ ]:
from numpyro.distributions import constraints
from numpyro.distributions.constraints import Constraint
from numpyro.distributions.transforms import Transform
from jax.tree_util import register_pytree_node_class
from numpyro.infer.reparam import ExplicitReparam
from numpyro import handlers

In [ ]:
# @register_pytree_node_class
class FlowReparam(Transform):
    r"""
    """

    domain = constraints.real
    
    def __init__(
        self, flow_object
    ):
        self.flow_object = flow_object
        
    def __call__(self, x):
        return self.flow_object.forward_pass(x)

    def _inverse(self, y):
        return self.flow_object.backward_pass(y)

    def log_abs_det_jacobian(self, x, y, intermediates):
        _, total_logdet = self.flow_object.forward_pass_with_logdet(x)
        return total_logdet

    def tree_flatten(self):
        # flow_object is the dynamic part
        return ((self.flow_object,), None)

    @classmethod
    def tree_unflatten(cls, aux_data, children):
        return cls(*children)

In [ ]:
flow_prior = dist.TransformedDistribution(
    dist.Normal(0,1).expand([chain.shape[-1]]).to_event(1),
    FlowReparam(mm)
)

In [ ]:
def model():
    z = numpyro.sample("z", dist.Normal(0,1).expand([chain.shape[-1]]).to_event(1))
    x, total_logdet = mm.forward_pass_with_logdet(z)
    numpyro.factor('lnprob', mm.log_prob(x) + 0.5 * jnp.sum(z**2) + total_logdet)
    numpyro.deterministic('x', x)
    
kernel = NUTS(model)
mcmc = MCMC(kernel, num_warmup=1000, num_samples=1000, num_chains=1)
mcmc.run(jrandom.key(2))

In [ ]:
x = mcmc.get_samples()['x']
x.shape

In [ ]:
plt.hist(gen_samps[:, 23], **hist_settings)
plt.hist(x[:, 0, 23],  **hist_settings)
plt.show()

In [ ]:
from enterprise_extensions import blocks
from enterprise.signals import signal_base, gp_signals
from enterprise.pulsar import Pulsar
from astropy.time import TimeDelta
from pint.simulation import make_fake_toas_fromMJDs
import astropy.units as u
from pint.models import get_model

from enterprise_extensions import sampler
from enterprise.signals import parameter
from enterprise.signals import gp_priors

In [ ]:
s = gp_signals.TimingModel(use_svd=True)
s += blocks.white_noise_block(
    vary=True,
    inc_ecorr=True,
    gp_ecorr=False,
    select='backend',
    tnequad=False,
)

# log10_A = parameter.Uniform(-18., -11.)('log10A')
# gamma = parameter.Uniform(0., 7.)('gamma')

# s+= gp_signals.FourierBasisGP(
#     spectrum=gp_priors.powerlaw(log10_A=log10_A, gamma = gamma),
#     modes = sig_unc.freqs,
#     # components = len(sig_unc.freqs),
#     Tspan=Tspan,
#     name="red_noise",
# )
log10_rho = parameter.Uniform(-9., -2., size=len(sig_unc.freqs))
s+= gp_signals.FourierBasisGP(
    spectrum=gp_priors.free_spectrum(log10_rho=log10_rho),
    components=len(sig_unc.freqs),
    modes = sig_unc.freqs,
    Tspan=Tspan,
    # name="red_noise",
)

pta = signal_base.PTA(
    [s(p) for p in psrs], signal_base.LogLikelihoodDenseCholesky
)


In [ ]:
# pta.set_default_params(noise_dict)

In [ ]:
from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc

In [ ]:
# ent_samp = sampler.setup_sampler(
#     pta,
#     # x0_user = np.hstack([p.sample() for p in pta.params]),
#     outdir="/data/taylor_group/Nima/PandorasBox/Trash/TrashCan",
#     resume=False,
#     empirical_distr=None,
#     groups=None,
#     human=None,
#     save_ext_dists=False,
#     loglkwargs={},
#     logpkwargs={},
# )
# set initial parameters drawn from prior
x0 = np.hstack([p.sample() for p in pta.params])
ndim = len(x0)
cov = np.diag(np.ones(ndim) * 0.01**2)
ent_samp = ptmcmc(ndim, pta.get_lnlikelihood, pta.get_lnprior, cov, 
                 outDir='/data/taylor_group/Nima/PandorasBox/Trash/GibbsTrashCan', resume=False)

In [ ]:
ent_samp.sample(
    p0 = np.hstack([p.sample() for p in pta.params]),
    Niter = int(2e5),
    SCAMweight=30,
    AMweight=15,
    DEweight=50,
)

In [ ]:
chain = np.loadtxt('/data/taylor_group/Nima/PandorasBox/Trash/GibbsTrashCan/chain_1.txt')
chain = chain[int(0.25 * chain.shape[0]):, :-4]
chain.shape

In [ ]:
for fidx in range(6, 36):
    plt.hist(chain[:, fidx], **hist_settings)
    plt.hist(samples[:, fidx - 6], **hist_settings)
    plt.title(pta.param_names[fidx])
    plt.show()